In [10]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [11]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [12]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Please generate 3 objects. Respond with raw JSON array only, no markdown fences, no explanation.
Example shape:
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    }
]
"""
    messages = []
    add_user_message(messages, prompt)
    text = chat(messages)
    # Strip fences in case the model adds them anyway
    text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text)

In [13]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [14]:
# Function to grade a test case + output using a model
import re

def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Respond with raw JSON only — no markdown, no fences, no explanation.
Use this exact shape:
{{
    "strengths": ["...", "..."],
    "weaknesses": ["...", "..."],
    "reasoning": "...",
    "score": 8
}}
"""
    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages, temperature=0.0)
    eval_text = eval_text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    # Fix invalid JSON backslash escapes (e.g. \d, \s from regex snippets)
    eval_text = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', eval_text)
    return json.loads(eval_text)

In [15]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [16]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [17]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [18]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7.666666666666667


In [19]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS CloudFormation S3 Bucket Parser\n\nI'll help you parse a CloudFormation template and extract S3 bucket logical IDs. Here's a solution:\n\n```python\nimport json\nimport yaml\nfrom typing import List, Union\n\ndef extract_s3_bucket_ids(template_content: Union[str, dict]) -> List[str]:\n    \"\"\"\n    Extract all S3 bucket logical IDs from a CloudFormation template.\n    \n    Args:\n        template_content: Either a JSON/YAML string or a parsed dictionary\n        \n    Returns:\n        List of logical IDs for resources with Type: AWS::S3::Bucket\n    \"\"\"\n    \n    # Parse template if it's a string\n    if isinstance(template_content, str):\n        try:\n            template = json.loads(template_content)\n        except json.JSONDecodeError:\n            template = yaml.safe_load(template_content)\n    else:\n        template = template_content\n    \n    # Extract S3 bucket logical IDs\n    s3_buckets = []\n    \n    if isinstance(template, dict) and